# Imports

In [11]:
import boto3
import os
from sagemaker.core import image_uris
from sagemaker.core.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.functions import Join
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig, SourceCode

# Initialization

In [12]:
def list_all_sagemaker_domains(region_name='eu-west-1'):
    """
    List all SageMaker domains in the current AWS account and region.

    Returns:
        list: A list of dictionaries containing domain details.
    """
    client = boto3.client('sagemaker',region_name=region_name)
    response = client.list_domains()
    return response.get('Domains', [])

def get_sagemaker_domain_arn(domain_name, region_name='eu-west-1'):
    """
    Get the ARN of a specific SageMaker domain by its name.

    Args:
        domain_name (str): The name of the SageMaker domain.
        region_name (str): The AWS region where the domain is located.

    Returns:
        str: The ARN of the specified SageMaker domain, or None if not found.
    """
    domains = list_all_sagemaker_domains(region_name)
    for domain in domains:
        if domain['DomainName'] == domain_name:
            return domain['DomainArn']
    return None

In [13]:
list_of_domains = list_all_sagemaker_domains()
print("List of SageMaker Domains:", list_of_domains)
print("ARN of the first domain:", get_sagemaker_domain_arn(list_of_domains[0]['DomainName']) if list_of_domains else "No domains found")

List of SageMaker Domains: [{'DomainArn': 'arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij', 'DomainId': 'd-mvykom11e1ij', 'DomainName': 'QuickSetupDomain-20260924T193773', 'Status': 'InService', 'CreationTime': datetime.datetime(2026, 9, 24, 16, 37, 48, 627000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 9, 24, 16, 38, 5, 153000, tzinfo=tzlocal()), 'Url': 'https://d-mvykom11e1ij.studio.eu-west-1.sagemaker.aws'}]
ARN of the first domain: arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij


In [14]:
pipeline_session = PipelineSession()
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')
print(f"Region: {region}, Account ID: {account_id}")
role = f"arn:aws:iam::746760141372:role/service-role/AmazonSageMaker-ExecutionRole-20260924T193773"
domain_name = os.getenv('DOMAIN_NAME') 
print(f"Domain Name from environment variable: {domain_name}")
domain_arn = get_sagemaker_domain_arn(domain_name) if domain_name else None
print(f"Domain ARN: {domain_arn if domain_arn else 'Domain not found'}")
print(f"role: {role}")

Region: eu-west-1, Account ID: 746760141372
Domain Name from environment variable: QuickSetupDomain-20260924T193773
Domain ARN: arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij
role: arn:aws:iam::746760141372:role/service-role/AmazonSageMaker-ExecutionRole-20260924T193773


# Pipeline Parameters

In [15]:
# S3 data locations
raw_data_s3_uri = ParameterString(
    name="RawDataS3Uri",
    default_value="s3://driftwatch-bucket/data/raw/telco_churn.csv",
)
processed_data_s3_uri = ParameterString(
    name="ProcessedDataS3Uri",
    default_value="s3://driftwatch-bucket/data/processed",
)
processed_model_s3_uri = ParameterString(
    name="ProcessedModelS3Uri",
    default_value="s3://driftwatch-bucket/models",
)

# Processor

In [16]:
script_processor = ScriptProcessor(
    command=["python3"],
    role=role,
    image_uri=image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.2-1",
        instance_type="ml.t3.medium",
    ),
    instance_count=1,
    instance_type="ml.t3.medium",
    base_job_name="driftwatch-processing-job",
    sagemaker_session=pipeline_session,
)

[10/06/26 21:08:06] INFO     Defaulting to only available Python version: py3                     ]8;id=3084004;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3084005;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

# Step 1 (Preprocessing)

In [17]:
step_preprocessing = ProcessingStep(
    name="PreprocessingStep",
    step_args=script_processor.run(
        code="steps/preprocess/preprocess.py",
        arguments=[
            "--input_s3_uri", raw_data_s3_uri,
            "--region", region,
            "--output_dir", "/opt/ml/processing/output",
        ],
        outputs=[
            ProcessingOutput(
                output_name="train",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "train"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/train",
                ),
            ),
            ProcessingOutput(
                output_name="validation",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "validate"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/validate",
                ),
            ),
            ProcessingOutput(
                output_name="test",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "test"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/test",
                ),
            ),
            ProcessingOutput(
                output_name="baseline",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "baseline"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/baseline",
                ),
            ),
        ],
    ),
)

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


# Step 2 (Training)

In [18]:
training_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    instance_type="ml.m5.large",
)

model_trainer = ModelTrainer(
    role=role,
    sagemaker_session=pipeline_session,
    base_job_name="driftwatch-training-job",
    source_code=SourceCode(
        source_dir="steps/train",
        entry_script="train.py",
    ),
    training_image=training_image,
    compute=Compute(
        instance_type="ml.m5.large",
        instance_count=1,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=Join(on="/", values=[processed_model_s3_uri, "artifacts"]),
    ),
    hyperparameters={
        "train-path": "/opt/ml/input/data/train/train.csv",
        "validation-path": "/opt/ml/input/data/validation/validation.csv",
        "model-dir": "/opt/ml/model",
    },
)

step_training = TrainingStep(
    name="TrainingStep",
    step_args=model_trainer.train(
        input_data_config=[
            InputData(
                channel_name="train",
                data_source=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
                    "train"
                ].S3Output.S3Uri,
            ),
            InputData(
                channel_name="validation",
                data_source=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
                    "validation"
                ].S3Output.S3Uri,
            ),
        ],
    ),
)

[10/06/26 21:08:07] INFO     Ignoring unnecessary instance type: ml.m5.large.                     ]8;id=3084010;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3084011;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=3084016;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3084017;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=3084022;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3084023;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=3084028;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=3084029;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             141502667606.dkr.ecr.eu-west-1.amazonaws.com/sagemaker-xgboost:1.                     
                             7-1                                                                                   

# Step 3 (Evaluation)

In [19]:
# Step 3 (Evaluation)
evaluation_processor = ScriptProcessor(
    command=["python3"],
    role=role,
    image_uri=image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.2-1",
        instance_type="ml.t3.medium",
    ),
    instance_count=1,
    instance_type="ml.t3.medium",
    base_job_name="driftwatch-evaluation-job",
    sagemaker_session=pipeline_session,
)

evaluation_output_path = Join(
    on="/", values=[processed_model_s3_uri, "evaluation"]
)

step_evaluation = ProcessingStep(
    name="EvaluationStep",
    step_args=evaluation_processor.run(
        code="steps/evaluate/evaluate.py",
        inputs=[
            ProcessingInput(
                input_name="test",
                s3_input=ProcessingS3Input(
                    s3_uri=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
                        "test"
                    ].S3Output.S3Uri,
                    s3_data_type="S3Prefix",
                    s3_input_mode="File",
                    local_path="/opt/ml/processing/input/test",
                ),
            ),
            ProcessingInput(
                input_name="model",
                s3_input=ProcessingS3Input(
                    s3_uri=step_training.properties.ModelArtifacts.S3ModelArtifacts,
                    s3_data_type="S3Prefix",
                    s3_input_mode="File",
                    local_path="/opt/ml/processing/input/model",
                ),
            ),
        ],
        arguments=[
            "--test-path", "/opt/ml/processing/input/test",
            "--model-dir", "/opt/ml/processing/input/model",
            "--output-dir", "/opt/ml/processing/output/evaluation",
        ],
        outputs=[
            ProcessingOutput(
                output_name="evaluation",
                s3_output=ProcessingS3Output(
                    s3_uri=evaluation_output_path,
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/evaluation",
                ),
            ),
        ],
    ),
)

                    INFO     Defaulting to only available Python version: py3                     ]8;id=3084034;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3084035;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

# Pipeline Creation

In [20]:
pipeline_name = "DriftWatchPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        raw_data_s3_uri,
        processed_data_s3_uri,
        processed_model_s3_uri,
    ],
    steps=[step_preprocessing],
    sagemaker_session=pipeline_session,
)

In [21]:
pipeline.upsert(
    role_arn=role,
    tags = [
        {"Key": "sagemaker:domain-arn", "Value": domain_arn},
    ]
)

[10/06/26 21:08:08] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=3084040;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3084041;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=3084046;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3084047;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

{'PipelineArn': 'arn:aws:sagemaker:eu-west-1:746760141372:pipeline/DriftWatchPipeline',
 'PipelineVersionId': 9,
 'ResponseMetadata': {'RequestId': '9ddedfac-83c0-47b6-b128-f46809b79612',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '9ddedfac-83c0-47b6-b128-f46809b79612',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '108',
   'date': 'Tue, 06 Oct 2026 21:08:08 GMT'},
  'RetryAttempts': 0}}

In [ ]:
execution = pipeline.start()
execution.wait()
execution_description = execution.describe()
execution_status = execution_description["PipelineExecutionStatus"]
print(f"Pipeline execution status: {execution_status}")
if execution_status != "Succeeded":
    raise RuntimeError(
        f"DriftWatchPipeline execution failed with status: {execution_status}"
    )